In [2]:
import pandas as pd
df = pd.read_excel('C:/Develop/深圳42/data/group_anlysis.xlsx')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 289386 entries, 0 to 289385
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   主订单编号   289386 non-null  int64 
 1   用户ID    289386 non-null  object
 2   付款时间    289386 non-null  object
 3   实付金额    289386 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 8.8+ MB


In [4]:
df.head()

,主订单编号,用户ID,付款时间,实付金额
0,73465136654,uid135460366,2023-01-01 09:32:12,166
1,73465136655,uid135460367,2023-01-01 09:11:50,117
2,73465136656,uid135460368,2023-01-01 11:49:02,166
3,73465136657,uid135460369,2023-01-01 12:20:24,77
4,73465136658,uid135460370,2023-01-01 01:23:15,158


In [5]:
df.describe()

,主订单编号,实付金额
count,2.893860e+05,289386.000000
mean,7.346531e+10,157.228857
std,1.022321e+05,98.872567
min,7.346514e+10,40.000000
25%,7.346522e+10,77.000000
50%,7.346530e+10,143.000000
75%,7.346540e+10,196.000000
max,7.346549e+10,581.000000


In [7]:
df['年月标签'] = df['付款时间'].str[:7] # 字符串截取, 获取年月字符串数据

In [10]:
df['年月标签'].value_counts().sort_index()

2023-01    12039
2023-02     4114
2023-03    18172
2023-04    12320
2023-05    15738
2023-06    44265
2023-07    16102
2023-08    28817
2023-09    60376
2023-10    21639
2023-11    40204
2023-12    15600
Name: 年月标签, dtype: int64

- 每个月要知道当前月份的新增用户  如果我们有用户注册的表, 知道了用户的注册时间, 直接可以计算了
- 在这里我们把用户的首次购买作为 新增的标志
- 计算当前月份的新增用户, 在后面的月份是否有购买  有购买算复购 

## 以2023年2月数据为例, 先算出一个月的数据来, 再for循环计算其它月份的

In [11]:
from pandas import DataFrame

month='2023-02'
sample:DataFrame=df.loc[df['年月标签']==month]
sample

,主订单编号,用户ID,付款时间,实付金额,年月标签
1889,73465138785,uid135460903,2023-02-01 11:27:36,174,2023-02
1890,73465138786,uid135461902,2023-02-01 12:22:35,196,2023-02
1915,73465138814,uid135461924,2023-02-02 20:24:53,235,2023-02
1921,73465138820,uid135461927,2023-02-03 11:35:04,235,2023-02
1960,73465138865,uid135460481,2023-02-05 13:49:07,193,2023-02
...,...,...,...,...,...
39007,73465180827,uid135478560,2023-02-28 19:16:15,43,2023-02
39008,73465180828,uid135479071,2023-02-28 19:22:05,146,2023-02
39009,73465180829,uid135479071,2023-02-28 19:22:05,308,2023-02
39010,73465180830,uid135483504,2023-02-28 23:06:16,77,2023-02


In [15]:
sample.shape

(4114, 5)

In [17]:
# 从2月的销售流水中去重得到所有2月的用户ID的唯一值
sample_unique = sample.drop_duplicates(subset=['用户ID'])

In [18]:
# 获取1月的数据
history_df = df.loc[df['年月标签']=='2023-01'] 
# 判断2月的用户是否在1月的用户数据中, 如果在数据中说明是1月的留存(复购)用户, 如果不在1月的用户数据中, 说明是2月的新增用户
sample_unique['用户ID'].isin(history_df['用户ID'])

1889      True
1890     False
1915     False
1921     False
1960      True
         ...  
39006    False
39007     True
39008     True
39010    False
39022    False
Name: 用户ID, Length: 3313, dtype: bool

In [21]:
# ~ 取反的符号  True →False  False →True
# 对在1月出现的ID范围内的数据取反 得到的就是不在这个范围的, 就是2月的新用户
sample_unique_new = sample_unique.loc[~(sample_unique['用户ID'].isin(history_df['用户ID']))]

In [22]:
# 二月的新增用户
sample_unique_new

,主订单编号,用户ID,付款时间,实付金额,年月标签
1890,73465138786,uid135461902,2023-02-01 12:22:35,196,2023-02
1915,73465138814,uid135461924,2023-02-02 20:24:53,235,2023-02
1921,73465138820,uid135461927,2023-02-03 11:35:04,235,2023-02
1961,73465138866,uid135461961,2023-02-05 15:35:28,193,2023-02
1972,73465138878,uid135461971,2023-02-06 12:06:45,174,2023-02
...,...,...,...,...,...
39004,73465180824,uid135483501,2023-02-28 18:39:23,373,2023-02
39005,73465180825,uid135483502,2023-02-28 18:52:44,148,2023-02
39006,73465180826,uid135483503,2023-02-28 22:37:59,43,2023-02
39010,73465180830,uid135483504,2023-02-28 23:06:16,77,2023-02


In [27]:
month_list = df['年月标签'].unique().tolist()[2:]
result_list = []
for month in month_list:
    # 取出一个月的数据 
    next_month_df = df.loc[df['年月标签']==month]
    # 新增用户的ID 出现在后面月份的数据中, 说明是复购用户
    retention_users_df = sample_unique_new.loc[sample_unique_new['用户ID'].isin(next_month_df['用户ID'])]
    # 把复购用户数量保存在列表中
    result_list.append([month+'留存情况',retention_users_df.shape[0]])

In [28]:
result_list

[['2023-03留存情况', 558],
 ['2023-04留存情况', 340],
 ['2023-05留存情况', 379],
 ['2023-06留存情况', 587],
 ['2023-07留存情况', 293],
 ['2023-08留存情况', 317],
 ['2023-09留存情况', 267],
 ['2023-10留存情况', 205],
 ['2023-11留存情况', 304],
 ['2023-12留存情况', 112]]

In [29]:
result_list.insert(0,['2023年2月新增用户:',sample_unique_new.shape[0]])

In [30]:
result_list

[['2023年2月新增用户:', 2740],
 ['2023-03留存情况', 558],
 ['2023-04留存情况', 340],
 ['2023-05留存情况', 379],
 ['2023-06留存情况', 587],
 ['2023-07留存情况', 293],
 ['2023-08留存情况', 317],
 ['2023-09留存情况', 267],
 ['2023-10留存情况', 205],
 ['2023-11留存情况', 304],
 ['2023-12留存情况', 112]]

In [36]:
df['年月标签'].unique()

array(['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06',
       '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12'],
      dtype=object)

In [38]:
list1 = [1,2,3,4,5]
list2= [6,7,8,9,10]
list(zip(list1,list2))

[(1, 6), (2, 7), (3, 8), (4, 9), (5, 10)]

In [41]:
list_0 = [0,0,0,0,0,0,0,0,0,0,0,0]
list1 = [5,6,7,8,9,10,11]
list2 = [1,2,3,4,5,6,7,8,9,10,11]
list(zip(list1,list2))

[(5, 1), (6, 2), (7, 3), (8, 4), (9, 5), (10, 6), (11, 7)]

In [55]:
for j,cnt in zip(list1,list2):
    print(j,cnt)

5 1
6 2
7 3
8 4
9 5
10 6
11 7


In [53]:
month_list = df['年月标签'].unique().tolist() # 获取所有月份的列表
final_df = pd.DataFrame() # 准备一个空白的df 用来保存最终的结果
for i in range(len(month_list)-1): # 一共计算11个月
    # 准备一个空白的列表, 用来保存当前月份计算的结果
    count_list = [0]*len(month_list)
    # 外层循环的目的是为了找到每个月的新增用户
    # 先筛选当前月份的数据
    target_month_df = df.loc[df['年月标签']==month_list[i]]
    target_month_df.drop_duplicates(subset=['用户ID'],inplace = True)
    # 如果是第一个月, 不需要判断了,所有的用户都是新用户
    if i==0:
        new_users_df = target_month_df.copy()
    else:
        # 如果不是1月, 2月以后得数据, 需要先获取前面的所有月份的数据  month_list[:i]
        history_df = df.loc[df['年月标签'].isin(month_list[:i])]
        # 判断当前的用户是否在前面的月份出现过, 如果没有出现过 就留下来, 是当前月份的新用户
        new_users_df = target_month_df.loc[(target_month_df['用户ID'].isin(history_df['用户ID']))==False]
    
    # 把新用户的数量保存到列表的第一个元素中
    count_list[0]=new_users_df.shape[0]
    #print(count_list)
    # 内层循环, 用来计算新用户在后面月份的复购情况
    for j,cnt in zip(range(i+1,len(month_list)),range(1,len(month_list))):
        # j 用来循环后面的月份, 从i+1开始, i指向2月份, j就是3月份
        # cnt 用来记录结果的 不管是哪个月份都是从1开始, 第0个元素记录的是新增用户
        next_month_df = df.loc[df['年月标签']==month_list[j]]
        # new_users_df['用户ID'].isin(next_month_df['用户ID']) 是True/False组成的列表 sum求和 计算的是True的数量
        retention_count =(new_users_df['用户ID'].isin(next_month_df['用户ID'])).sum()
        # 保存结果到列表
        count_list[cnt] = retention_count
    # 如果不是第一个月, 需要和历史的月份进行判断, 如果在历史月份中出现过,就不是新用户
    # 要统计的是在前面的月份中,没有出现过的用户ID
    result=pd.DataFrame({month_list[i]:count_list}).T
    final_df = pd.concat([final_df,result])
final_df.columns = ['当月新增','+1月','+2月','+3月','+4月','+5月','+6月','+7月','+8月','+9月','+10月','+11月']

C:\Users\Administrator\AppData\Local\Temp\ipykernel_14652\373218438.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_month_df.drop_duplicates(subset=['用户ID'],inplace = True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_14652\373218438.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_month_df.drop_duplicates(subset=['用户ID'],inplace = True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_14652\373218438.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guid

In [54]:
final_df

,当月新增,+1月,+2月,+3月,+4月,+5月,+6月,+7月,+8月,+9月,+10月,+11月
2023-01,8193,573,1601,1050,1079,1906,815,1102,863,628,1049,372
2023-02,2740,558,340,379,587,293,317,267,205,304,112,0
2023-03,8753,1176,1232,2112,799,1032,777,616,1064,360,0,0
2023-04,5859,828,1208,502,618,482,329,526,171,0,0,0
2023-05,6912,1575,626,747,464,392,569,198,0,0,0,0
2023-06,16458,1575,1775,1153,923,1482,496,0,0,0,0,0
2023-07,6514,801,404,257,390,144,0,0,0,0,0,0
2023-08,11781,1030,606,813,331,0,0,0,0,0,0,0
2023-09,30214,2206,2482,971,0,0,0,0,0,0,0,0
2023-10,11253,1147,452,0,0,0,0,0,0,0,0,0


In [84]:
month_list = df['年月标签'].unique().tolist() # 获取所有月份的列表
final_df = pd.DataFrame() # 准备一个空白的df 用来保存最终的结果
for i in range(len(month_list)-1): # 一共计算11个月
    # 准备一个空白的列表, 用来保存当前月份计算的结果
    count_list = []
    # 外层循环的目的是为了找到每个月的新增用户
    # 先筛选当前月份的数据
    target_month_df = df.loc[df['年月标签']==month_list[i]]
    target_month_df.drop_duplicates(subset=['用户ID'],inplace = True)
    # 如果是第一个月, 不需要判断了,所有的用户都是新用户
    if i==0:
        new_users_df = target_month_df.copy()
    else:
        # 如果不是1月, 2月以后得数据, 需要先获取前面的所有月份的数据  month_list[:i]
        history_df = df.loc[df['年月标签'].isin(month_list[:i])]
        # 判断当前的用户是否在前面的月份出现过, 如果没有出现过 就留下来, 是当前月份的新用户
        new_users_df = target_month_df.loc[(target_month_df['用户ID'].isin(history_df['用户ID']))==False]
    
    # 把新用户的数量保存到列表里
    count_list.append(new_users_df.shape[0])
    #print(count_list)
    # 内层循环, 用来计算新用户在后面月份的复购情况
    for j in range(i+1,len(month_list)):
        # j 用来循环后面的月份, 从i+1开始, i指向2月份, j就是3月份
        # cnt 用来记录结果的 不管是哪个月份都是从1开始, 第0个元素记录的是新增用户
        next_month_df = df.loc[df['年月标签']==month_list[j]]
        # new_users_df['用户ID'].isin(next_month_df['用户ID']) 是True/False组成的列表 sum求和 计算的是True的数量
        retention_count =(new_users_df['用户ID'].isin(next_month_df['用户ID'])).sum()
        # 保存结果到列表
        count_list.append(retention_count)
    # 如果不是第一个月, 需要和历史的月份进行判断, 如果在历史月份中出现过,就不是新用户
    # 要统计的是在前面的月份中,没有出现过的用户ID
    result=pd.DataFrame({month_list[i]:count_list}).T
    final_df = pd.concat([final_df,result])
final_df.fillna(0,inplace=True)
final_df.columns=['当月新增','+1月','+2月','+3月','+4月','+5月','+6月','+7月','+8月','+9月','+10月','+11月']

C:\Users\Administrator\AppData\Local\Temp\ipykernel_14652\3755026837.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_month_df.drop_duplicates(subset=['用户ID'],inplace = True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_14652\3755026837.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_month_df.drop_duplicates(subset=['用户ID'],inplace = True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_14652\3755026837.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_g

In [76]:
result = final_df.divide(final_df['当月新增'],axis=0).iloc[:,1:]

In [77]:
result.insert(loc=0,column='当月新增',value=final_df['当月新增'])

In [78]:
result

,当月新增,+1月,+2月,+3月,+4月,+5月,+6月,+7月,+8月,+9月,+10月,+11月
2023-01,8193,0.069938,0.195411,0.128158,0.131698,0.232638,0.099475,0.134505,0.105334,0.076651,0.128036,0.045405
2023-02,2740,0.203650,0.124088,0.138321,0.214234,0.106934,0.115693,0.097445,0.074818,0.110949,0.040876,0.000000
2023-03,8753,0.134354,0.140752,0.241289,0.091283,0.117902,0.088770,0.070376,0.121558,0.041129,0.000000,0.000000
2023-04,5859,0.141321,0.206179,0.085680,0.105479,0.082267,0.056153,0.089776,0.029186,0.000000,0.000000,0.000000
2023-05,6912,0.227865,0.090567,0.108073,0.067130,0.056713,0.082321,0.028646,0.000000,0.000000,0.000000,0.000000
2023-06,16458,0.095698,0.107850,0.070057,0.056082,0.090047,0.030137,0.000000,0.000000,0.000000,0.000000,0.000000
2023-07,6514,0.122966,0.062020,0.039453,0.059871,0.022106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2023-08,11781,0.087429,0.051439,0.069009,0.028096,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2023-09,30214,0.073013,0.082147,0.032137,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2023-10,11253,0.101928,0.040167,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [88]:
# 客单价 
month_list = df['年月标签'].unique().tolist() # 获取所有月份的列表
final_df_monetary = pd.DataFrame() # 准备一个空白的df 用来保存最终的结果
for i in range(len(month_list)-1): # 一共计算11个月
    # 准备一个空白的列表, 用来保存当前月份计算的结果
    count_list = []
    # 外层循环的目的是为了找到每个月的新增用户
    # 先筛选当前月份的数据
    target_month_df = df.loc[df['年月标签']==month_list[i]]
    target_month_df.drop_duplicates(subset=['用户ID'],inplace = True)
    # 如果是第一个月, 不需要判断了,所有的用户都是新用户
    if i==0:
        new_users_df = target_month_df.copy()
    else:
        # 如果不是1月, 2月以后得数据, 需要先获取前面的所有月份的数据  month_list[:i]
        history_df = df.loc[df['年月标签'].isin(month_list[:i])]
        # 判断当前的用户是否在前面的月份出现过, 如果没有出现过 就留下来, 是当前月份的新用户
        new_users_df = target_month_df.loc[(target_month_df['用户ID'].isin(history_df['用户ID']))==False]
    
    # 把新用户的数量保存到列表里
    count_list.append(new_users_df.shape[0])
    #print(count_list)
    # 内层循环, 用来计算新用户在后面月份的复购情况
    for j in range(i+1,len(month_list)):
        # j 用来循环后面的月份, 从i+1开始, i指向2月份, j就是3月份
        # cnt 用来记录结果的 不管是哪个月份都是从1开始, 第0个元素记录的是新增用户
        next_month_df = df.loc[df['年月标签']==month_list[j]]
        # new_users_df['用户ID'].isin(next_month_df['用户ID']) 是True/False组成的列表 sum求和 计算的是True的数量
        # retention_count =(new_users_df['用户ID'].isin(next_month_df['用户ID'])).sum()
        total_pay = next_month_df.loc[next_month_df['用户ID'].isin(new_users_df['用户ID']),'实付金额'].sum()
        # 保存结果到列表
        count_list.append(total_pay)
    # 如果不是第一个月, 需要和历史的月份进行判断, 如果在历史月份中出现过,就不是新用户
    # 要统计的是在前面的月份中,没有出现过的用户ID
    result=pd.DataFrame({month_list[i]:count_list}).T
    final_df_monetary = pd.concat([final_df_monetary,result])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_14652\2265249913.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_month_df.drop_duplicates(subset=['用户ID'],inplace = True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_14652\2265249913.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_month_df.drop_duplicates(subset=['用户ID'],inplace = True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_14652\2265249913.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/use

In [89]:
final_df_monetary.fillna(0,inplace=True)
final_df_monetary.columns =['当月新增','+1月','+2月','+3月','+4月','+5月','+6月','+7月','+8月','+9月','+10月','+11月']

In [90]:
final_df_monetary

,当月新增,+1月,+2月,+3月,+4月,+5月,+6月,+7月,+8月,+9月,+10月,+11月
2023-01,8193,152038,474019.0,321004.0,301462.0,616853.0,185251.0,292858.0,262933.0,180543.0,287445.0,93318.0
2023-02,2740,151328,94338.0,105943.0,189951.0,55792.0,77418.0,77279.0,59372.0,78433.0,28066.0,0.0
2023-03,8753,289928,284522.0,605839.0,162036.0,255507.0,228685.0,162398.0,275678.0,92286.0,0.0,0.0
2023-04,5859,179413,323887.0,103464.0,143104.0,141508.0,85408.0,144100.0,41001.0,0.0,0.0,0.0
2023-05,6912,395880,128426.0,160008.0,123942.0,94661.0,146627.0,46841.0,0.0,0.0,0.0,0.0
2023-06,16458,300293,383712.0,312063.0,233662.0,369007.0,108013.0,0.0,0.0,0.0,0.0,0.0
2023-07,6514,162437,97194.0,57777.0,84881.0,31884.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-08,11781,276063,143451.0,186071.0,80948.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-09,30214,555657,548818.0,277662.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-10,11253,232308,124408.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [91]:
final_df_monetary/final_df

,当月新增,+1月,+2月,+3月,+4月,+5月,+6月,+7月,+8月,+9月,+10月,+11月
2023-01,1.0,265.336824,296.076827,305.718095,279.390176,323.637461,227.301840,265.751361,304.673233,287.488854,274.018112,250.854839
2023-02,1.0,271.197133,277.464706,279.532982,323.596252,190.416382,244.220820,289.434457,289.619512,258.003289,250.589286,NaN
2023-03,1.0,246.537415,230.943182,286.855587,202.798498,247.584302,294.317889,263.633117,259.095865,256.350000,NaN,NaN
2023-04,1.0,216.682367,268.118377,206.103586,231.559871,293.585062,259.598784,273.954373,239.771930,NaN,NaN,NaN
2023-05,1.0,251.352381,205.153355,214.200803,267.116379,241.482143,257.692443,236.570707,NaN,NaN,NaN,NaN
2023-06,1.0,190.662222,216.175775,270.653079,253.154930,248.992578,217.768145,NaN,NaN,NaN,NaN,NaN
2023-07,1.0,202.792759,240.579208,224.813230,217.643590,221.416667,NaN,NaN,NaN,NaN,NaN,NaN
2023-08,1.0,268.022330,236.717822,228.869619,244.555891,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-09,1.0,251.884406,221.119259,285.954686,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-10,1.0,202.535310,275.238938,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [87]:
final_df

,当月新增,+1月,+2月,+3月,+4月,+5月,+6月,+7月,+8月,+9月,+10月,+11月
2023-01,8193,573,1601.0,1050.0,1079.0,1906.0,815.0,1102.0,863.0,628.0,1049.0,372.0
2023-02,2740,558,340.0,379.0,587.0,293.0,317.0,267.0,205.0,304.0,112.0,0.0
2023-03,8753,1176,1232.0,2112.0,799.0,1032.0,777.0,616.0,1064.0,360.0,0.0,0.0
2023-04,5859,828,1208.0,502.0,618.0,482.0,329.0,526.0,171.0,0.0,0.0,0.0
2023-05,6912,1575,626.0,747.0,464.0,392.0,569.0,198.0,0.0,0.0,0.0,0.0
2023-06,16458,1575,1775.0,1153.0,923.0,1482.0,496.0,0.0,0.0,0.0,0.0,0.0
2023-07,6514,801,404.0,257.0,390.0,144.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-08,11781,1030,606.0,813.0,331.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-09,30214,2206,2482.0,971.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-10,11253,1147,452.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
